In [11]:
import anndata as ad
import numpy as np
from pathlib import Path
import torch
from tqdm import tqdm

import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

from MethylCDM.models.betaVAE import BetaVAE

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_PATH = "/ddn_exa/campbell/sli/methylcdm-project/models/beta_vae/betaVAE_sweep_20260306_133219/trial_72/best-epoch=151-val_loss=1.3515.ckpt"

In [8]:
model = BetaVAE.load_from_checkpoint(CHECKPOINT_PATH, map_location=DEVICE)
model.eval()
model.to(DEVICE)

BetaVAE(
  (encoder): MethylEncoder(
    (encoder): Sequential(
      (0): Dropout(p=0.21443160444238737, inplace=False)
      (1): Sequential(
        (0): Linear(in_features=211572, out_features=2048, bias=True)
        (1): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (2): GELU(approximate='none')
      )
      (2): Sequential(
        (0): Linear(in_features=2048, out_features=512, bias=True)
        (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (2): GELU(approximate='none')
      )
      (3): Sequential(
        (0): Linear(in_features=512, out_features=128, bias=True)
        (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (2): GELU(approximate='none')
      )
      (4): Sequential(
        (0): Linear(in_features=128, out_features=128, bias=True)
        (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (2): GELU(approximate='none')
      )
    )
  )
  (z_mu): Linear(in_features=128, out_features=128, bia

In [3]:
adata = ad.read_h5ad("/ddn_exa/campbell/sli/methylcdm-project/data/training/methylation/pancancer_cohort_adata.h5ad")

In [9]:
OUTPUT_BASE = Path("/ddn_exa/campbell/sli/methylcdm-project/data/embeddings")

In [15]:
for i, (idx, row) in enumerate(tqdm(adata.obs.iterrows(), total = adata.n_obs)):
    barcode = row['barcode']
    file_id = idx.upper()
    project = row['project_id']

    project_dir = OUTPUT_BASE / project
    project_dir.mkdir(exist_ok = True, parents = True)

    sample_data = adata[i, :].X
    if not isinstance(sample_data, np.ndarray):
        sample_data = sample_data.toarray() if hasattr(sample_data, "toarray") else np.array(sample_data)
    sample_tensor = torch.tensor(sample_data, dtype = torch.float32)

    embedding = generate_embeddings(model, sample_tensor)
    filename = f"{barcode}.{file_id}.npy"
    np.save(project_dir / filename, embedding)

  0%|                                        | 2/1898 [00:19<5:06:34,  9.70s/it]


KeyboardInterrupt: 

In [14]:
def generate_embeddings(model, sample_tensor):
    """
    Computes beta-VAE latente embeddings on a single sample tensor.
    """

    sample_tesnor = sample_tensor.to(DEVICE).unsqueeze(0)
    with torch.no_grad():
        z_mu, _, _ = model.encode(sample_tensor)
    return z_mu.squeeze(0).cpu().numpy()